<a href="https://colab.research.google.com/github/edina-s/ADVANCED-CSS/blob/master/01_Uvod_v_podatkovno_znanost/notebooks/01_prvi_stik_s_podatki/A_prvi_stik_s_podatki.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Prvi stik s podatki — A: prikaz (spletna trgovina)
**Akademija UI za poslovne aplikacije — Modul 1, Python ekosistem in spoznavanje s surovimi podatki**

Ta zvezek poganja predavatelj. Vajo naredite v zvezku **`B_prvi_stik_s_podatki`** (telekom).

Celico zaženete s **Shift + Enter**.

### Ponovitev
- **DataFrame** = tabela: vrstica je opazovanje, stolpec je spremenljivka. En stolpec je **Series**.
- **Vektorizacija**: operacijo zapišemo za cel stolpec naenkrat — brez `for` zanke.
- Štirje ukazi za vsak nov nabor podatkov: `pd.read_csv()`, `.shape`, `.head()`, `.info()` / `.dtypes`.
- **Tip v Pandasu ni isto kot tip po pomenu.** Pandas ne ve, da je poštna številka oznaka in da je `odhod` skupina.
  Pogoste napake: datum kot besedilo (`object` / `str`), poštna številka kot število, celo število kot `float`.
- Cilj: **ena vrstica = eno opazovanje** (stranka, naročilo, …), en stolpec = ena lastnost.

### Nastavitev okolja
Ta celica poskrbi, da Python najde datoteko `utils/orodja.py` s pomožnimi funkcijami.
V Google Colabu najprej prenese gradivo z GitHuba. V naslednjih zvezkih je ista celica skrita.

In [1]:
# Nastavitev okolja
import subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/valira-ai/ui-za-poslovne-aplikacije.git"

if "google.colab" in sys.modules:           # Google Colab: prenesemo gradivo z GitHuba
    KOREN = Path("/content/ui-za-poslovne-aplikacije")
    if not KOREN.exists():
        subprocess.run(["git", "clone", "-q", REPO_URL, str(KOREN)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "plotnine>=0.15", "kagglehub"], check=True)
else:                                       # lokalno: poiščemo mapo, v kateri je utils/orodja.py
    KOREN = Path.cwd()
    while not (KOREN / "utils/orodja.py").exists() and KOREN != KOREN.parent:
        KOREN = KOREN.parent
sys.path.insert(0, str(KOREN))              # da Python najde utils/orodja.py
print("Gradivo je v:", KOREN)

Gradivo je v: /content/ui-za-poslovne-aplikacije


## A0. Prenos podatkov s Kaggla
Oba nabora podatkov sta javno objavljena na Kagglu. Knjižnica `kagglehub` ju prenese (prijava ni potrebna)
in vrne mapo, kamor ju je shranila. Ob drugem zagonu ju vzame iz predpomnilnika.

In [2]:
import kagglehub

trgovina_pot = Path(kagglehub.dataset_download("olistbr/brazilian-ecommerce"))
telekom_pot = Path(kagglehub.dataset_download("abhinav89/telecom-customer"))

print(trgovina_pot)
sorted(f.name for f in trgovina_pot.glob("*.csv"))

Using Colab cache for faster access to the 'brazilian-ecommerce' dataset.


100%|██████████| 13.6M/13.6M [00:00<00:00, 204MB/s]

Extracting files...


/kaggle/input/brazilian-ecommerce


['olist_customers_dataset.csv',
 'olist_geolocation_dataset.csv',
 'olist_order_items_dataset.csv',
 'olist_order_payments_dataset.csv',
 'olist_order_reviews_dataset.csv',
 'olist_orders_dataset.csv',
 'olist_products_dataset.csv',
 'olist_sellers_dataset.csv',
 'product_category_name_translation.csv']

Spletna trgovina ima **9 povezanih tabel**. Telekom ima eno samo tabelo — a koliko stolpcev?

In [3]:
import pandas as pd

telekom_surovo = pd.read_csv(telekom_pot / "Telecom_customer churn.csv")
print(telekom_surovo.shape)
telekom_surovo.columns[:20].tolist()        # prvih 20 imen stolpcev

(100000, 100)


['rev_Mean',
 'mou_Mean',
 'totmrc_Mean',
 'da_Mean',
 'ovrmou_Mean',
 'ovrrev_Mean',
 'vceovr_Mean',
 'datovr_Mean',
 'roam_Mean',
 'change_mou',
 'change_rev',
 'drop_vce_Mean',
 'drop_dat_Mean',
 'blck_vce_Mean',
 'blck_dat_Mean',
 'unan_vce_Mean',
 'unan_dat_Mean',
 'plcd_vce_Mean',
 'plcd_dat_Mean',
 'recv_vce_Mean']

**100 stolpcev** s kratkimi angleškimi imeni (`mou_Mean`, `totmrc_Mean`, `eqpdays` …).
Za delavnico smo izbrali **22 stolpcev** in jim dali slovenska imena.
Seznam je v `utils/orodja.py` kot slovar `TELEKOM_STOLPCI`: slovensko ime → (izvirno ime, opis).

In [4]:
from utils.orodja import TELEKOM_STOLPCI

nova_imena = {izvirno: slovensko for slovensko, (izvirno, opis) in TELEKOM_STOLPCI.items()}
telekom = telekom_surovo[list(nova_imena)].rename(columns=nova_imena)   # izberemo in preimenujemo
print(telekom.shape)
telekom.head(3)

(100000, 22)


,id_stranke,odhod,mesecev_narocnine,povp_mesecni_prihodek,povp_mesecna_narocnina,povp_minute,povp_minute_6m,sprememba_minut,povp_minute_prekoracitev,povp_gostovanje,...,stevilo_narocnin,stevilo_telefonov,dni_z_napravo,cena_naprave,splet_na_napravi,dualband,regija,zakonski_stan,dohodek_razred,stevilo_avtomobilov
0,1000001,1,61,23.9975,22.500,219.25,322.0,-157.25,0.00,0.0,...,2,2.0,361.0,149.98999,WCMB,Y,NORTHWEST/ROCKY MOUNTAIN AREA,S,4.0,3.0
1,1000002,0,56,57.4925,37.425,482.75,477.0,532.25,22.75,0.0,...,1,7.0,240.0,NaN,WC,N,CHICAGO AREA,S,5.0,1.0
2,1000003,1,58,16.9900,16.990,10.25,11.0,-4.25,0.00,0.0,...,1,2.0,1504.0,29.98999,NaN,N,GREAT LAKES AREA,M,5.0,2.0


Vse to — prenos, izbor stolpcev in shranjevanje v mapo `data/` — smo zapakirali v **funkcijo**
`poskrbi_za_podatke()` v `utils/orodja.py`. V vseh naslednjih zvezkih jo samo pokličemo:

In [5]:
from utils.orodja import poskrbi_za_podatke
DATA = poskrbi_za_podatke()

Prenašam abhinav89/telecom-customer s Kaggla ...
Prenašam olistbr/brazilian-ecommerce s Kaggla ...
Using Colab cache for faster access to the 'brazilian-ecommerce' dataset.
Podatki so pripravljeni v: /content/ui-za-poslovne-aplikacije/data


---
# Del A — Prikaz: spletna trgovina
Podatki brazilske spletne trgovine Olist: ~100 000 naročil (2016–2018), razdeljenih v več povezanih tabel
(naročila, postavke, plačila, stranke, izdelki, ocene).

## A1. Knjižnice

In [7]:
import numpy as np
import pandas as pd

## A2. Naložimo tabelo naročil
`pd.read_csv` prebere CSV datoteko v **DataFrame** (tabelo).

In [8]:
narocila = pd.read_csv(DATA / "trgovina" / "olist_orders_dataset.csv")
narocila.shape          # (število vrstic, število stolpcev)

(99441, 8)

In [9]:
narocila.head()         # prvih 5 vrstic

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00


In [11]:
narocila.info()         # tipi stolpcev in število ne-manjkajočih vrednosti

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   order_id                       99441 non-null  object
 1   customer_id                    99441 non-null  object
 2   order_status                   99441 non-null  object
 3   order_purchase_timestamp       99441 non-null  object
 4   order_approved_at              99281 non-null  object
 5   order_delivered_carrier_date   97658 non-null  object
 6   order_delivered_customer_date  96476 non-null  object
 7   order_estimated_delivery_date  99441 non-null  object
dtypes: object(8)
memory usage: 6.1+ MB


**Kaj opazimo?**
- Vsi datumi (`order_purchase_timestamp`, …) so tipa `object` / `str` — torej **besedilo**, ne datum.
  Z besedilom ne moremo računati (npr. koliko dni je trajala dostava).
- Nekateri datumi imajo manj ne-manjkajočih vrednosti kot ima tabela vrstic → **manjkajoče vrednosti** (popoldne).

## A3. Popravimo tip: besedilo → datum

In [12]:
datumi = ["order_purchase_timestamp", "order_approved_at", "order_delivered_carrier_date",
          "order_delivered_customer_date", "order_estimated_delivery_date"]
narocila = pd.read_csv(DATA / "trgovina" / "olist_orders_dataset.csv", parse_dates=datumi)
narocila.dtypes

,0
order_id,object
customer_id,object
order_status,object
order_purchase_timestamp,datetime64[ns]
order_approved_at,datetime64[ns]
order_delivered_carrier_date,datetime64[ns]
order_delivered_customer_date,datetime64[ns]
order_estimated_delivery_date,datetime64[ns]


Zdaj lahko z datumi računamo — na primer, koliko dni je trajala dostava.
Ta stolpec bomo kasneje tudi **napovedovali** (regresija).

In [14]:
narocila["dni_dostave"] = (narocila["order_delivered_customer_date"]
                           - narocila["order_purchase_timestamp"]).dt.days
narocila["dni_dostave"].describe()

,dni_dostave
count,96476.000000
mean,12.094086
std,9.551746
min,0.000000
25%,6.000000
50%,10.000000
75%,15.000000
max,209.000000


## A4. Številka, ki ni številka: poštna številka

In [23]:
stranke = pd.read_csv(DATA / "trgovina" / "olist_customers_dataset.csv")
stranke.dtypes

,0
customer_id,object
customer_unique_id,object
customer_zip_code_prefix,int64
customer_city,object
customer_state,object


In [24]:
stranke.head(3)

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP


Kaj v resnici piše v datoteki? Preberimo prve 3 vrstice kot navadno besedilo:

In [25]:
with open(DATA / "trgovina" / "olist_customers_dataset.csv", encoding="utf-8") as f:
    for _ in range(3):
        print(f.readline().strip())

"customer_id","customer_unique_id","customer_zip_code_prefix","customer_city","customer_state"
"06b8999e2fba1a1fbc88172c00ba8bc7","861eff4711a542e4b93843c6dd7febb0","14409",franca,SP
"18955e83d337fd6b2def6b18a428ac77","290c77bc529b7ac935b93aa66c333dc3","09790",sao bernardo do campo,SP


V datoteki piše `"09790"`, Pandas pa je prebral število `9790` — **vodilna ničla je izginila**.
Poštna številka je **oznaka**, ne količina: z njo ne računamo povprečja.
Pandasu povemo, naj stolpec prebere kot besedilo:

In [26]:
stranke = pd.read_csv(DATA / "trgovina" / "olist_customers_dataset.csv",
                      dtype={"customer_zip_code_prefix": str})
stranke.head(3)

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,09790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,01151,sao paulo,SP


**Kaj je ena vrstica?** Trgovina ob vsakem naročilu ustvari **nov** `customer_id`.
Isto osebo poveže šele `customer_unique_id`:

In [27]:
print("Vrstic:                       ", len(stranke))
print("Različnih customer_id:        ", stranke["customer_id"].nunique())
print("Različnih customer_unique_id: ", stranke["customer_unique_id"].nunique())

Vrstic:                        99441
Različnih customer_id:         99441
Različnih customer_unique_id:  96096


Ena vrstica = en `customer_id` = eno naročilo. Za zgodovino nakupov ene osebe (npr. kasneje pri segmentaciji strank)
moramo uporabiti `customer_unique_id`. **Pri vsaki tabeli najprej preverimo, kaj pomeni ena vrstica.**

## A5. Zakaj ne `for` zanke? Vektorizacija
Ceni vsake postavke prištejemo 22 % DDV — enkrat z zanko, enkrat z enim ukazom nad celim stolpcem.

In [29]:
import time
postavke = pd.read_csv(DATA / "trgovina" / "olist_order_items_dataset.csv")
print(f"Število postavk: {len(postavke):,}   različnih naročil: {postavke['order_id'].nunique():,}")

zacetek = time.perf_counter()
z_ddv = []
for cena in postavke["price"]:          # Python zanka: vrstico za vrstico
    z_ddv.append(cena * 1.22)
cas_zanka = time.perf_counter() - zacetek

zacetek = time.perf_counter()
z_ddv_vekt = postavke["price"] * 1.22   # vektorizirano: cel stolpec naenkrat
cas_vekt = time.perf_counter() - zacetek

print(f"Zanka:          {cas_zanka * 1000:8.2f} ms")
print(f"Vektorizirano:  {cas_vekt * 1000:8.2f} ms   (~{cas_zanka / cas_vekt:.0f}x hitreje)")

Število postavk: 112,650   različnih naročil: 98,666
Zanka:             22.06 ms
Vektorizirano:      0.66 ms   (~34x hitreje)


Mimogrede: postavk je več kot naročil. **Ena vrstica = ena postavka**, eno naročilo ima lahko več postavk.
Ko tabele združujemo, moramo postavke najprej sešteti po naročilu.

## A6. Iz pol-strukturiranih podatkov v tabelo
Sistemi pogosto beležijo dogodke v obliki **JSON** — vsak zapis ima lahko druga polja.
`pd.json_normalize` jih razgrne v tabelo. Kjer polja ni, dobimo `NaN`.

In [30]:
dogodki = [
    {"id_stranke": 1001, "cas": "2026-11-02 08:15", "tip": "klic",
     "podrobnosti": {"trajanje_s": 125, "smer": "odhodni"}},
    {"id_stranke": 1001, "cas": "2026-11-02 12:40", "tip": "sms",
     "podrobnosti": {"dolzina_znakov": 42}},
    {"id_stranke": 1002, "cas": "2026-11-03 19:05", "tip": "klic",
     "podrobnosti": {"trajanje_s": 610, "smer": "dohodni", "gostovanje": True}},
]
pd.json_normalize(dogodki)

,id_stranke,cas,tip,podrobnosti.trajanje_s,podrobnosti.smer,podrobnosti.dolzina_znakov,podrobnosti.gostovanje
0,1001,2026-11-02 08:15,klic,125.0,odhodni,NaN,NaN
1,1001,2026-11-02 12:40,sms,NaN,NaN,42.0,NaN
2,1002,2026-11-03 19:05,klic,610.0,dohodni,NaN,True


## A7. Iz nestrukturiranih podatkov v značilko
Komentar kupca je prosto besedilo. Najpreprostejša pretvorba v število: **dolžina komentarja**.
(Naprednejše: kategorija pritožbe, sentiment, … )

In [32]:
ocene = pd.read_csv(DATA / "trgovina" / "olist_order_reviews_dataset.csv")
ocene["dolzina_komentarja"] = ocene["review_comment_message"].str.len()
ocene[["review_score", "review_comment_message", "dolzina_komentarja"]].dropna().head()

,review_score,review_comment_message,dolzina_komentarja
3,5,Recebi bem antes do prazo estipulado.,37.0
4,5,Parabéns lojas lannister adorei comprar pela I...,100.0
9,4,aparelho eficiente. no site a marca do aparelh...,174.0
12,4,"Mas um pouco ,travando...pelo valor ta Boa.\r\n",45.0
15,5,"Vendedor confiável, produto ok e entrega antes...",56.0


---
Naprej: zvezek **`B_prvi_stik_s_podatki`** — isto naredite sami na podatkih telekoma.